
**Step 1: Business Case**

**Project Title**

**RAG-Based HR Policy Assistant for Employee Knowledge Support**

**Business Problem**

Employees often struggle to find accurate answers about company policies because information is spread across multiple HR documents. A regular LLM may give a fluent answer, but it may guess or hallucinate because it does not know the company’s internal policies.
Proposed Solution
Build a RAG-based HR assistant that retrieves relevant sections from HR policy documents before generating an answer. This helps make the response more accurate, grounded, and traceable.
Users
Employees, HR staff, managers, and new hires.
Example Questions Users May Ask

1.	How many vacation days do employees receive?
2.	What is the remote work policy?
3.	What is the sick leave policy?
4.	How does the company handle overtime?
5.	What benefits are available to full-time employees?

**Why Retrieval Is Necessary**

Retrieval is necessary because company policies are specific to the organization. A general LLM may not know the exact rules, but a RAG system can search the provided HR documents and answer based on the retrieved text.
What Problems LLM-Only Systems Would Face
A normal LLM may:

•	Guess policy details

•	Give outdated or generic answers

•	Mix information from different companies

•	Fail to show where the answer came from

So, our selected use case is:

**An HR Policy Q&A Assistant that helps employees ask natural language questions and receive grounded answers from internal HR policy documents.**




**Step 2**

**Creating the HR Knowledge Base**

In [ ]:
# ================================
# STEP 2: DATA & KNOWLEDGE BASE
# RAG-Based HR Policy Assistant
# 15 HR Policy Documents
# ================================

import os
import re

# Create folder for HR documents
os.makedirs("hr_documents", exist_ok=True)

documents = {
    "leave_policy.txt": """
    Employee Leave Policy

    Full-time employees receive 15 paid vacation days per year after completing 90 days of employment.
    Part-time employees may receive prorated leave depending on their weekly schedule.
    Vacation requests should be submitted at least two weeks in advance through the HR portal.
    Managers approve or deny leave requests based on staffing needs.
    Unused vacation days may be carried over up to a maximum of 5 days into the next calendar year.
    """,

    "sick_leave_policy.txt": """
    Sick Leave Policy

    Employees receive 8 paid sick days per year.
    Sick leave may be used for personal illness, medical appointments, or caring for an immediate family member.
    Employees should notify their manager as soon as possible if they are unable to work due to illness.
    If an employee is absent for more than three consecutive workdays, HR may request medical documentation.
    """,

    "remote_work_policy.txt": """
    Remote Work Policy

    Eligible employees may work remotely up to two days per week with manager approval.
    Remote work is available only for roles that can be performed effectively outside the office.
    Employees working remotely must be available during core business hours from 9:00 AM to 4:00 PM.
    Company data must only be accessed through approved devices and secure company systems.
    """,

    "overtime_policy.txt": """
    Overtime Policy

    Non-exempt employees are eligible for overtime pay when they work more than 40 hours in a workweek.
    Overtime must be approved by a manager before it is worked.
    Approved overtime is paid at one and one-half times the employee's regular hourly rate.
    Exempt employees are not eligible for overtime pay.
    Employees must accurately record all hours worked in the timekeeping system.
    """,

    "benefits_policy.txt": """
    Employee Benefits Policy

    Full-time employees are eligible for medical, dental, and vision insurance after 60 days of employment.
    The company offers a retirement savings plan with employer matching contributions.
    Employees may also be eligible for life insurance, disability coverage, and wellness programs.
    Benefit enrollment must be completed during onboarding or annual open enrollment.
    """,

    "code_of_conduct.txt": """
    Workplace Code of Conduct

    Employees are expected to act with honesty, respect, professionalism, and integrity.
    Harassment, discrimination, bullying, and retaliation are not tolerated.
    Employees should report workplace concerns to their manager, HR, or the ethics hotline.
    Confidential company information must not be shared with unauthorized individuals.
    """,

    "payroll_policy.txt": """
    Payroll Policy

    Employees are paid biweekly through direct deposit.
    Employees must submit accurate time records before the payroll deadline.
    Payroll corrections should be reported to HR or payroll support as soon as possible.
    Bonuses, overtime, and deductions are processed according to company policy and applicable law.
    Employees can access pay statements through the employee self-service portal.
    """,

    "performance_review_policy.txt": """
    Performance Review Policy

    Employees receive formal performance reviews once per year.
    Managers may also provide informal feedback throughout the year.
    Performance reviews evaluate goal achievement, teamwork, communication, and job responsibilities.
    Employees may create development goals with their manager.
    Poor performance may result in a performance improvement plan.
    """,

    "training_development_policy.txt": """
    Training and Development Policy

    The company encourages employees to participate in job-related training.
    New employees must complete required onboarding training within their first 30 days.
    Employees may request approval for professional development courses.
    Training may include compliance modules, technical skills, leadership development, and safety instruction.
    Managers are responsible for supporting employee growth and learning.
    """,

    "employee_onboarding_policy.txt": """
    Employee Onboarding Policy

    New employees must complete onboarding paperwork before or during their first week.
    Onboarding includes identity verification, tax forms, benefits information, company policies, and system access.
    New hires receive an introduction to their department, manager, and team members.
    HR provides guidance on company expectations and available employee resources.
    """,

    "termination_policy.txt": """
    Employee Termination Policy

    Employment may end through resignation, retirement, layoff, or termination.
    Employees who resign should provide at least two weeks of notice when possible.
    Company property must be returned before the employee's final day.
    Final pay is processed according to company policy and applicable law.
    HR may conduct an exit interview to collect feedback.
    """,

    "workplace_safety_policy.txt": """
    Workplace Safety Policy

    Employees are expected to follow all workplace safety rules and procedures.
    Unsafe conditions, accidents, or injuries should be reported immediately to a manager or HR.
    Employees must use required protective equipment when applicable.
    The company provides safety training for roles with physical or operational risks.
    Violations of safety rules may result in corrective action.
    """,

    "data_security_policy.txt": """
    Data Security Policy

    Employees must protect company, customer, and employee data.
    Confidential information should only be accessed by authorized employees.
    Passwords must not be shared with coworkers or external individuals.
    Employees must report suspected data breaches or suspicious activity immediately.
    Company devices should be locked when unattended.
    """,

    "travel_expense_policy.txt": """
    Travel and Expense Policy

    Employees may be reimbursed for approved business travel expenses.
    Travel must be approved by a manager before expenses are incurred.
    Employees should submit receipts and expense reports within 30 days of travel.
    Reimbursable expenses may include transportation, lodging, meals, and business-related supplies.
    Personal expenses are not reimbursable.
    """,

    "attendance_policy.txt": """
    Attendance and Punctuality Policy

    Employees are expected to report to work on time and follow their assigned schedules.
    Employees should notify their manager as early as possible if they will be late or absent.
    Repeated unexcused absences or lateness may result in corrective action.
    Approved leave, sick time, and emergency absences should be documented through HR systems.
    Good attendance supports team productivity and customer service.
    """
}

# Save each document as a .txt file
for filename, text in documents.items():
    filepath = os.path.join("hr_documents", filename)
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(text)

print("HR documents created successfully!")
print(f"Total files created: {len(os.listdir('hr_documents'))}")
print(os.listdir("hr_documents"))

HR documents created successfully!
Total files created: 15
['data_security_policy.txt', 'travel_expense_policy.txt', 'employee_onboarding_policy.txt', 'remote_work_policy.txt', 'termination_policy.txt', 'training_development_policy.txt', 'sick_leave_policy.txt', 'performance_review_policy.txt', 'leave_policy.txt', 'payroll_policy.txt', 'attendance_policy.txt', 'workplace_safety_policy.txt', 'code_of_conduct.txt', 'benefits_policy.txt', 'overtime_policy.txt']


In [ ]:
# ================================
# LOAD AND CLEAN DOCUMENTS
# ================================

def clean_text(text):
    """
    Cleans and normalizes raw text.
    """
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text


def load_documents(folder_path):
    """
    Loads all .txt documents from a folder.
    Returns filename, raw text, and cleaned text.
    """
    loaded_docs = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            filepath = os.path.join(folder_path, filename)

            with open(filepath, "r", encoding="utf-8") as file:
                raw_text = file.read()

            cleaned_text = clean_text(raw_text)

            loaded_docs.append({
                "filename": filename,
                "raw_text": raw_text,
                "cleaned_text": cleaned_text
            })

    return loaded_docs


docs = load_documents("hr_documents")

print(f"Total documents loaded: {len(docs)}")

for doc in docs:
    print("\n----------------------")
    print("File:", doc["filename"])
    print("Preview:", doc["cleaned_text"][:250])

Total documents loaded: 15

----------------------
File: data_security_policy.txt
Preview: data security policy employees must protect company, customer, and employee data. confidential information should only be accessed by authorized employees. passwords must not be shared with coworkers or external individuals. employees must report sus

----------------------
File: travel_expense_policy.txt
Preview: travel and expense policy employees may be reimbursed for approved business travel expenses. travel must be approved by a manager before expenses are incurred. employees should submit receipts and expense reports within 30 days of travel. reimbursabl

----------------------
File: employee_onboarding_policy.txt
Preview: employee onboarding policy new employees must complete onboarding paperwork before or during their first week. onboarding includes identity verification, tax forms, benefits information, company policies, and system access. new hires receive an intro

--------------------

**Step 3 - Chunking Strategies.**

We have choosen 2 chunking strategies:

1.Paragraph-based chunking

2. Sliding-window chunking


In [ ]:
# ================================
# STEP 3: CHUNKING STRATEGIES
# ================================

import re

def paragraph_chunking(docs):
    """
    Splits each document into paragraph-based chunks.
    Each paragraph becomes one chunk.
    """
    chunks = []

    for doc in docs:
        filename = doc["filename"]
        raw_text = doc["raw_text"]

        # Split by blank lines
        paragraphs = raw_text.split("\n\n")

        for i, paragraph in enumerate(paragraphs):
            cleaned_paragraph = clean_text(paragraph)

            # Avoid empty chunks
            if len(cleaned_paragraph) > 20:
                chunks.append({
                    "filename": filename,
                    "chunk_id": f"{filename}_paragraph_{i}",
                    "chunking_strategy": "paragraph",
                    "text": cleaned_paragraph
                })

    return chunks


def sliding_window_chunking(docs, window_size=40, overlap=10):
    """
    Splits each document into fixed-size word chunks with overlap.
    window_size = number of words per chunk
    overlap = number of words repeated between chunks
    """
    chunks = []

    for doc in docs:
        filename = doc["filename"]
        text = doc["cleaned_text"]
        words = text.split()

        start = 0
        chunk_number = 0

        while start < len(words):
            end = start + window_size
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)

            if len(chunk_text) > 20:
                chunks.append({
                    "filename": filename,
                    "chunk_id": f"{filename}_sliding_{chunk_number}",
                    "chunking_strategy": "sliding_window",
                    "text": chunk_text
                })

            start += window_size - overlap
            chunk_number += 1

    return chunks

In [ ]:
# Create chunks using both strategies
paragraph_chunks = paragraph_chunking(docs)
sliding_chunks = sliding_window_chunking(docs, window_size=40, overlap=10)

# Combine all chunks into one list
all_chunks = paragraph_chunks + sliding_chunks

print("Paragraph chunks:", len(paragraph_chunks))
print("Sliding window chunks:", len(sliding_chunks))
print("Total chunks:", len(all_chunks))

Paragraph chunks: 25
Sliding window chunks: 34
Total chunks: 59


In [ ]:
# Preview first 5 chunks
for chunk in all_chunks[:5]:
    print("\n----------------------")
    print("Filename:", chunk["filename"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Strategy:", chunk["chunking_strategy"])
    print("Text:", chunk["text"])


----------------------
Filename: data_security_policy.txt
Chunk ID: data_security_policy.txt_paragraph_1
Strategy: paragraph
Text: employees must protect company, customer, and employee data. confidential information should only be accessed by authorized employees. passwords must not be shared with coworkers or external individuals. employees must report suspected data breaches or suspicious activity immediately. company devices should be locked when unattended.

----------------------
Filename: travel_expense_policy.txt
Chunk ID: travel_expense_policy.txt_paragraph_0
Strategy: paragraph
Text: travel and expense policy

----------------------
Filename: travel_expense_policy.txt
Chunk ID: travel_expense_policy.txt_paragraph_1
Strategy: paragraph
Text: employees may be reimbursed for approved business travel expenses. travel must be approved by a manager before expenses are incurred. employees should submit receipts and expense reports within 30 days of travel. reimbursable expenses may

**Why We Chose These Two Strategies**

**Paragraph-Based Chunking**

Paragraph chunking keeps related policy information together. This is useful for HR documents because each paragraph often explains one policy rule.

**Strength**: Preserves meaning and context.

**Weakness**: Some chunks may be too broad or contain extra information.

**Sliding-Window Chunking**

Sliding-window chunking splits text into fixed word groups with overlap. This helps avoid losing important information at chunk boundaries.

**Strength**: Good for retrieval because overlapping words preserve context.

**Weakness**: It can create repeated information and more chunks.

**Step 4: Embeddings + Retrieval.**

In [ ]:
!pip install sentence-transformers

In [ ]:
# ================================
# STEP 4: EMBEDDINGS + RETRIEVAL
# ================================

import numpy as np
from sentence_transformers import SentenceTransformer

# Load an open-source embedding model
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [ ]:
# Extract text from all chunks
chunk_texts = [chunk["text"] for chunk in all_chunks]

# Generate embeddings
chunk_embeddings = embedding_model.encode(chunk_texts)

# Convert embeddings to NumPy array
chunk_embeddings = np.array(chunk_embeddings)

print("Total chunks embedded:", len(chunk_texts))
print("Embedding matrix shape:", chunk_embeddings.shape)

Total chunks embedded: 59
Embedding matrix shape: (59, 384)


In [ ]:
def cosine_similarity_manual(query_embedding, document_embeddings):
    """
    Manually computes cosine similarity between one query embedding
    and all document chunk embeddings.
    """
    query_norm = np.linalg.norm(query_embedding)
    document_norms = np.linalg.norm(document_embeddings, axis=1)

    dot_products = np.dot(document_embeddings, query_embedding)

    similarities = dot_products / (document_norms * query_norm)

    return similarities

In [ ]:
def retrieve_top_k_chunks(query, chunks, chunk_embeddings, top_k=3):
    """
    Retrieves the top-k most relevant chunks for a user query.
    """
    # Convert query into embedding
    query_embedding = embedding_model.encode(query)
    query_embedding = np.array(query_embedding)

    # Compute cosine similarity manually
    similarities = cosine_similarity_manual(query_embedding, chunk_embeddings)

    # Get indexes of top-k most similar chunks
    top_k_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_k_indices:
        results.append({
            "rank": len(results) + 1,
            "filename": chunks[idx]["filename"],
            "chunk_id": chunks[idx]["chunk_id"],
            "chunking_strategy": chunks[idx]["chunking_strategy"],
            "similarity_score": similarities[idx],
            "text": chunks[idx]["text"]
        })

    return results

In [ ]:
query = "How many sick days do employees receive?"

retrieved_chunks = retrieve_top_k_chunks(
    query=query,
    chunks=all_chunks,
    chunk_embeddings=chunk_embeddings,
    top_k=3
)

for result in retrieved_chunks:
    print("\n==============================")
    print("Rank:", result["rank"])
    print("File:", result["filename"])
    print("Strategy:", result["chunking_strategy"])
    print("Similarity Score:", round(result["similarity_score"], 4))
    print("Text:", result["text"])


Rank: 1
File: sick_leave_policy.txt
Strategy: paragraph
Similarity Score: 0.8399
Text: employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are unable to work due to illness. if an employee is absent for more than three consecutive workdays, hr may request medical documentation.

Rank: 2
File: sick_leave_policy.txt
Strategy: sliding_window
Similarity Score: 0.7626
Text: sick leave policy employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are

Rank: 3
File: leave_policy.txt
Strategy: sliding_window
Similarity Score: 0.5569
Text: employee leave policy full-time employees receive 15 paid vacation days per year after completing 90 days of employment. part-tim

In [ ]:
query = "Can employees work remotely?"
retrieved_chunks = retrieve_top_k_chunks(query, all_chunks, chunk_embeddings, top_k=3)

for result in retrieved_chunks:
    print("\n==============================")
    print("Rank:", result["rank"])
    print("File:", result["filename"])
    print("Strategy:", result["chunking_strategy"])
    print("Similarity Score:", round(result["similarity_score"], 4))
    print("Text:", result["text"])


Rank: 1
File: remote_work_policy.txt
Strategy: paragraph
Similarity Score: 0.6855
Text: eligible employees may work remotely up to two days per week with manager approval. remote work is available only for roles that can be performed effectively outside the office. employees working remotely must be available during core business hours from 9:00 am to 4:00 pm. company data must only be accessed through approved devices and secure company systems.

Rank: 2
File: remote_work_policy.txt
Strategy: sliding_window
Similarity Score: 0.6592
Text: remote work policy eligible employees may work remotely up to two days per week with manager approval. remote work is available only for roles that can be performed effectively outside the office. employees working remotely must be available during core

Rank: 3
File: remote_work_policy.txt
Strategy: sliding_window
Similarity Score: 0.5854
Text: the office. employees working remotely must be available during core business hours from 9:00 am to 4:00 p

In [ ]:
query = "When do employees get overtime pay?"
retrieved_chunks = retrieve_top_k_chunks(query, all_chunks, chunk_embeddings, top_k=3)

for result in retrieved_chunks:
    print("\n==============================")
    print("Rank:", result["rank"])
    print("File:", result["filename"])
    print("Strategy:", result["chunking_strategy"])
    print("Similarity Score:", round(result["similarity_score"], 4))
    print("Text:", result["text"])


Rank: 1
File: overtime_policy.txt
Strategy: paragraph
Similarity Score: 0.712
Text: non-exempt employees are eligible for overtime pay when they work more than 40 hours in a workweek. overtime must be approved by a manager before it is worked. approved overtime is paid at one and one-half times the employee's regular hourly rate. exempt employees are not eligible for overtime pay. employees must accurately record all hours worked in the timekeeping system.

Rank: 2
File: overtime_policy.txt
Strategy: sliding_window
Similarity Score: 0.6844
Text: overtime policy non-exempt employees are eligible for overtime pay when they work more than 40 hours in a workweek. overtime must be approved by a manager before it is worked. approved overtime is paid at one and one-half times the

Rank: 3
File: overtime_policy.txt
Strategy: sliding_window
Similarity Score: 0.6601
Text: approved overtime is paid at one and one-half times the employee's regular hourly rate. exempt employees are not eligible fo

In [ ]:
query = "What should employees do if company data is breached?"
retrieved_chunks = retrieve_top_k_chunks(query, all_chunks, chunk_embeddings, top_k=3)

for result in retrieved_chunks:
    print("\n==============================")
    print("Rank:", result["rank"])
    print("File:", result["filename"])
    print("Strategy:", result["chunking_strategy"])
    print("Similarity Score:", round(result["similarity_score"], 4))
    print("Text:", result["text"])


Rank: 1
File: data_security_policy.txt
Strategy: paragraph
Similarity Score: 0.6524
Text: employees must protect company, customer, and employee data. confidential information should only be accessed by authorized employees. passwords must not be shared with coworkers or external individuals. employees must report suspected data breaches or suspicious activity immediately. company devices should be locked when unattended.

Rank: 2
File: data_security_policy.txt
Strategy: sliding_window
Similarity Score: 0.6051
Text: data security policy employees must protect company, customer, and employee data. confidential information should only be accessed by authorized employees. passwords must not be shared with coworkers or external individuals. employees must report suspected data breaches or suspicious activity immediately.

Rank: 3
File: data_security_policy.txt
Strategy: sliding_window
Similarity Score: 0.5975
Text: employees must report suspected data breaches or suspicious activity immed

In [ ]:
test_queries = [
    "How many sick days do employees receive?",
    "Can employees work remotely?",
    "When do employees get overtime pay?",
    "What benefits are available to full-time employees?",
    "What should employees do if company data is breached?"
]

for query in test_queries:
    print("\n################################################")
    print("QUERY:", query)

    results = retrieve_top_k_chunks(query, all_chunks, chunk_embeddings, top_k=3)

    for result in results:
        print("\nRank:", result["rank"])
        print("File:", result["filename"])
        print("Strategy:", result["chunking_strategy"])
        print("Score:", round(result["similarity_score"], 4))
        print("Chunk:", result["text"][:250])


################################################
QUERY: How many sick days do employees receive?

Rank: 1
File: sick_leave_policy.txt
Strategy: paragraph
Score: 0.8399
Chunk: employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are unable to work due to illn

Rank: 2
File: sick_leave_policy.txt
Strategy: sliding_window
Score: 0.7626
Chunk: sick leave policy employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are

Rank: 3
File: leave_policy.txt
Strategy: sliding_window
Score: 0.5569
Chunk: employee leave policy full-time employees receive 15 paid vacation days per year after completing 90 days of employment. part-time employees may receive prorated leave de

**Step 5 is RAG Pipeline with LLM Integration.**

In [ ]:
# ================================
# STEP 5A: LOAD LLM WITHOUT PIPELINE
# ================================

!pip install transformers sentencepiece accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

llm_model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

print("LLM loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully!


In [ ]:
def generate_text(prompt, max_new_tokens=150):
    """
    Generates text directly from FLAN-T5 without using pipeline().
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def generate_llm_only_answer(query):
    """
    Generates an answer without retrieved context.
    This simulates a normal LLM-only response.
    """
    prompt = f"""
    Answer the following HR policy question based on general knowledge.

    Question: {query}

    Answer:
    """

    return generate_text(prompt, max_new_tokens=150)

In [ ]:
def generate_rag_answer(query, top_k=3):
    """
    Retrieves top-k relevant chunks and generates a grounded RAG answer.
    """
    retrieved_chunks = retrieve_top_k_chunks(
        query=query,
        chunks=all_chunks,
        chunk_embeddings=chunk_embeddings,
        top_k=top_k
    )

    context = "\n\n".join(
        [
            f"Source: {chunk['filename']}\nText: {chunk['text']}"
            for chunk in retrieved_chunks
        ]
    )

    prompt = f"""
    You are an HR policy assistant.

    Use only the provided HR policy context to answer the question.
    If the answer is not found in the context, say:
    The policy documents do not provide enough information to answer this question.

    HR Policy Context:
    {context}

    Question:
    {query}

    Grounded Answer:
    """

    answer = generate_text(prompt, max_new_tokens=150)

    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "rag_answer": answer
    }

In [ ]:
query = "How many sick days do employees receive?"

llm_only_answer = generate_llm_only_answer(query)
rag_result = generate_rag_answer(query, top_k=3)

print("QUESTION:")
print(query)

print("\n==============================")
print("LLM-ONLY ANSWER:")
print(llm_only_answer)

print("\n==============================")
print("RETRIEVED CONTEXT:")
for chunk in rag_result["retrieved_chunks"]:
    print("\nRank:", chunk["rank"])
    print("File:", chunk["filename"])
    print("Strategy:", chunk["chunking_strategy"])
    print("Similarity Score:", round(chunk["similarity_score"], 4))
    print("Text:", chunk["text"])

print("\n==============================")
print("RAG ANSWER:")
print(rag_result["rag_answer"])

QUESTION:
How many sick days do employees receive?

LLM-ONLY ANSWER:
a week

RETRIEVED CONTEXT:

Rank: 1
File: sick_leave_policy.txt
Strategy: paragraph
Similarity Score: 0.8399
Text: employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are unable to work due to illness. if an employee is absent for more than three consecutive workdays, hr may request medical documentation.

Rank: 2
File: sick_leave_policy.txt
Strategy: sliding_window
Similarity Score: 0.7626
Text: sick leave policy employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are

Rank: 3
File: leave_policy.txt
Strategy: sliding_window
Similarity Score: 0.5569
Text: employee leave policy full-time e

In [ ]:
# ================================
# STEP 5F: COMPARE LLM-ONLY VS RAG
# ================================

demo_queries = [
    "How many sick days do employees receive?",
    "Can employees work remotely?",
    "When do employees get overtime pay?",
    "What benefits are available to full-time employees?",
    "What should employees do if company data is breached?"
]

comparison_results = []

for query in demo_queries:
    llm_only_answer = generate_llm_only_answer(query)
    rag_result = generate_rag_answer(query, top_k=3)

    top_sources = [chunk["filename"] for chunk in rag_result["retrieved_chunks"]]

    comparison_results.append({
        "Question": query,
        "LLM_Only_Answer": llm_only_answer,
        "RAG_Answer": rag_result["rag_answer"],
        "Top_Retrieved_Sources": ", ".join(top_sources)
    })

# Display results
for result in comparison_results:
    print("\n====================================================")
    print("QUESTION:")
    print(result["Question"])

    print("\nLLM OUTPUT WITHOUT RETRIEVAL:")
    print(result["LLM_Only_Answer"])

    print("\nLLM OUTPUT WITH RAG:")
    print(result["RAG_Answer"])

    print("\nTOP RETRIEVED SOURCES:")
    print(result["Top_Retrieved_Sources"])


QUESTION:
How many sick days do employees receive?

LLM OUTPUT WITHOUT RETRIEVAL:
a week

LLM OUTPUT WITH RAG:
90

TOP RETRIEVED SOURCES:
sick_leave_policy.txt, sick_leave_policy.txt, leave_policy.txt

QUESTION:
Can employees work remotely?

LLM OUTPUT WITHOUT RETRIEVAL:
No

LLM OUTPUT WITH RAG:
Yes

TOP RETRIEVED SOURCES:
remote_work_policy.txt, remote_work_policy.txt, remote_work_policy.txt

QUESTION:
When do employees get overtime pay?

LLM OUTPUT WITHOUT RETRIEVAL:
a year

LLM OUTPUT WITH RAG:
overtime

TOP RETRIEVED SOURCES:
overtime_policy.txt, overtime_policy.txt, overtime_policy.txt

QUESTION:
What benefits are available to full-time employees?

LLM OUTPUT WITHOUT RETRIEVAL:
401k

LLM OUTPUT WITH RAG:
Benefits

TOP RETRIEVED SOURCES:
benefits_policy.txt, benefits_policy.txt, benefits_policy.txt

QUESTION:
What should employees do if company data is breached?

LLM OUTPUT WITHOUT RETRIEVAL:
if company data is breached

LLM OUTPUT WITH RAG:
Lock company devices

TOP RETRIEVED SOU

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame(comparison_results)
comparison_df

,Question,LLM_Only_Answer,RAG_Answer,Top_Retrieved_Sources
0,How many sick days do employees receive?,a week,90,"sick_leave_policy.txt, sick_leave_policy.txt, ..."
1,Can employees work remotely?,No,Yes,"remote_work_policy.txt, remote_work_policy.txt..."
2,When do employees get overtime pay?,a year,overtime,"overtime_policy.txt, overtime_policy.txt, over..."
3,What benefits are available to full-time emplo...,401k,Benefits,"benefits_policy.txt, benefits_policy.txt, bene..."
4,What should employees do if company data is br...,if company data is breached,Lock company devices,"data_security_policy.txt, data_security_policy..."


In [ ]:
comparison_df.to_csv("llm_vs_rag_comparison.csv", index=False)

print("Comparison saved as llm_vs_rag_comparison.csv")

Comparison saved as llm_vs_rag_comparison.csv


**Task 6 — Business Impact Analysis**

How RAG Improves Answer Reliability
Retrieval-Augmented Generation improves answer reliability by allowing the system to first search the HR document knowledge base before generating a response. Instead of depending only on the LLM’s general training knowledge, the RAG system retrieves relevant policy chunks and uses them as context. This makes the final answer more grounded, accurate, and traceable.
For example, when a user asks, “How many sick days do employees receive?”, the RAG system retrieves the sick leave policy and answers based on the specific company rule. This is more reliable than a normal LLM response because the answer comes from the organization’s internal policy document rather than general assumptions.
This directly supports the assignment goal of building a system that retrieves relevant document chunks, uses them as context, and generates grounded answers.

**Reduction in Hallucination Compared to LLM-Only System**

A major weakness of an LLM-only system is that it may generate fluent but incorrect answers. Since the LLM does not have direct access to internal HR documents, it may guess policy details such as the number of vacation days, sick days, or remote work limits.
The RAG system reduces hallucination by limiting the answer to retrieved HR policy content. In our implementation, the prompt instructs the model to use only the provided HR policy context. If the answer is not found, the system should say that the policy documents do not provide enough information.
This is important in a business setting because inaccurate HR answers can create confusion, employee dissatisfaction, compliance issues, or inconsistent policy interpretation.

**How Retrieval Improves Domain Specificity**

Retrieval improves domain specificity because it connects the LLM to company-specific documents. A general LLM may know common HR practices, but it does not know the exact rules of our sample organization.
For example, a general LLM might say employees usually receive sick leave based on company policy or local law. However, the RAG system can retrieve the exact policy stating that employees receive 8 paid sick days per year. It can also retrieve details about remote work, overtime, benefits, payroll, onboarding, data security, and workplace conduct.
This makes the system more useful for enterprise environments because employees need answers based on their actual company policies, not generic HR information.

**Failure Cases**

Although RAG improves reliability, it can still fail in several ways.

First, bad retrieval can happen when the system retrieves the wrong chunk. For example, if a user asks about paid leave but the system retrieves a sick leave or attendance policy instead, the generated answer may be incomplete or incorrect.

Second, irrelevant chunks can confuse the LLM. If the top-k retrieved chunks include unrelated policies, the model may mix information from different documents.

Third, poor chunking can reduce answer quality. If chunks are too small, they may lose important context. If chunks are too large, they may include too much unrelated information. This is why we compared paragraph-based chunking and sliding-window chunking.
Fourth, missing information can still be a problem. If the HR documents do not contain an answer, the system cannot produce a reliable answer. In that case, the best response is to clearly state that the policy documents do not provide enough information.

Finally, the system depends on the quality and freshness of the document store. If HR policies are outdated or incomplete, the RAG system may still return outdated or incomplete answers.

**Summary**

Overall, RAG improves business knowledge assistants by making answers more reliable, grounded, and specific to internal company documents. Compared to an LLM-only system, RAG reduces hallucination because the model uses retrieved policy chunks before answering. Retrieval also improves domain specificity by allowing the system to answer based on exact HR policies rather than general knowledge. However, RAG can still fail when retrieval returns irrelevant chunks, when chunking loses context, or when the document store does not contain the needed information.

** Running Demos**

In [ ]:
# ================================
# DEMO QUERY 1
# LLM-Only vs RAG Comparison
# ================================

demo_query_1 = "How many sick days do employees receive?"

# Generate LLM-only answer
llm_only_answer_1 = generate_llm_only_answer(demo_query_1)

# Generate RAG answer
rag_result_1 = generate_rag_answer(demo_query_1, top_k=3)

print("DEMO QUERY 1")
print("=" * 60)
print("Question:")
print(demo_query_1)

print("\nLLM OUTPUT WITHOUT RETRIEVAL:")
print(llm_only_answer_1)

print("\nRETRIEVED CONTEXT:")
for chunk in rag_result_1["retrieved_chunks"]:
    print("\nRank:", chunk["rank"])
    print("Source File:", chunk["filename"])
    print("Chunking Strategy:", chunk["chunking_strategy"])
    print("Similarity Score:", round(chunk["similarity_score"], 4))
    print("Retrieved Text:", chunk["text"])

print("\nLLM OUTPUT WITH RAG:")
print(rag_result_1["rag_answer"])

DEMO QUERY 1
Question:
How many sick days do employees receive?

LLM OUTPUT WITHOUT RETRIEVAL:
a week

RETRIEVED CONTEXT:

Rank: 1
Source File: sick_leave_policy.txt
Chunking Strategy: paragraph
Similarity Score: 0.8399
Retrieved Text: employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are unable to work due to illness. if an employee is absent for more than three consecutive workdays, hr may request medical documentation.

Rank: 2
Source File: sick_leave_policy.txt
Chunking Strategy: sliding_window
Similarity Score: 0.7626
Retrieved Text: sick leave policy employees receive 8 paid sick days per year. sick leave may be used for personal illness, medical appointments, or caring for an immediate family member. employees should notify their manager as soon as possible if they are

Rank: 3
Source File: leave_policy.txt
Chun

In [ ]:
# ================================
# DEMO QUERY 2
# LLM-Only vs RAG Comparison
# ================================

demo_query_2 = "Can employees work remotely?"

# Generate LLM-only answer
llm_only_answer_2 = generate_llm_only_answer(demo_query_2)

# Generate RAG answer
rag_result_2 = generate_rag_answer(demo_query_2, top_k=3)

print("DEMO QUERY 2")
print("=" * 60)
print("Question:")
print(demo_query_2)

print("\nLLM OUTPUT WITHOUT RETRIEVAL:")
print(llm_only_answer_2)

print("\nRETRIEVED CONTEXT:")
for chunk in rag_result_2["retrieved_chunks"]:
    print("\nRank:", chunk["rank"])
    print("Source File:", chunk["filename"])
    print("Chunking Strategy:", chunk["chunking_strategy"])
    print("Similarity Score:", round(chunk["similarity_score"], 4))
    print("Retrieved Text:", chunk["text"])

print("\nLLM OUTPUT WITH RAG:")
print(rag_result_2["rag_answer"])

DEMO QUERY 2
Question:
Can employees work remotely?

LLM OUTPUT WITHOUT RETRIEVAL:
No

RETRIEVED CONTEXT:

Rank: 1
Source File: remote_work_policy.txt
Chunking Strategy: paragraph
Similarity Score: 0.6855
Retrieved Text: eligible employees may work remotely up to two days per week with manager approval. remote work is available only for roles that can be performed effectively outside the office. employees working remotely must be available during core business hours from 9:00 am to 4:00 pm. company data must only be accessed through approved devices and secure company systems.

Rank: 2
Source File: remote_work_policy.txt
Chunking Strategy: sliding_window
Similarity Score: 0.6592
Retrieved Text: remote work policy eligible employees may work remotely up to two days per week with manager approval. remote work is available only for roles that can be performed effectively outside the office. employees working remotely must be available during core

Rank: 3
Source File: remote_work_policy.t

In [ ]:
# ================================
# DEMO SUMMARY TABLE
# ================================

import pandas as pd

demo_results = [
    {
        "Question": demo_query_1,
        "LLM Output Without Retrieval": llm_only_answer_1,
        "LLM Output With RAG": rag_result_1["rag_answer"],
        "Top Retrieved Source": rag_result_1["retrieved_chunks"][0]["filename"]
    },
    {
        "Question": demo_query_2,
        "LLM Output Without Retrieval": llm_only_answer_2,
        "LLM Output With RAG": rag_result_2["rag_answer"],
        "Top Retrieved Source": rag_result_2["retrieved_chunks"][0]["filename"]
    }
]

demo_df = pd.DataFrame(demo_results)
demo_df

,Question,LLM Output Without Retrieval,LLM Output With RAG,Top Retrieved Source
0,How many sick days do employees receive?,a week,90,sick_leave_policy.txt
1,Can employees work remotely?,No,Yes,remote_work_policy.txt


In [ ]:
demo_df.to_csv("rag_demo_results.csv", index=False)

print("Demo results saved as rag_demo_results.csv")

Demo results saved as rag_demo_results.csv
